In [ ]:
import datetime as dt

import geopandas as gpd
import pandas as pd

In [ ]:
spatial_extent_filepath = "../data/processed/bern-extent.gpkg"

ts_df_filepath = "../data/raw/bern-summer-2025-pcd.csv"
stations_gdf_filepath = "../data/raw/bern-metadata-2025.csv"

# ts_df_time_col = "Time_UTC"
stations_id_col = "Nr"
# stations_src_crs = "epsg:4326"
stations_src_crs = "epsg:2056"
stations_x_col = "LV_95_E"
stations_y_col = "LV_95_N"

# study period (e.g., JJA)
start_year = 2025
end_year = 2025
start_month = 6
end_month = 8

# output params
dst_crs = "epsg:2056"

# output files
dst_ts_df_filepath = "../data/interim/bern-lcd-ts-df.csv"
dst_stations_gdf_filepath = "../data/interim/bern-lcd-stations.gpkg"

In [ ]:
ts_df = pd.read_csv(ts_df_filepath, parse_dates=["time"], index_col="time")
# filter by study period
# note that we need to resample each year data before concat
# otherwise resample would add nan rows for all the missing non-JJA data
ts_df = pd.concat(
    [
        ts_df[
            pd.Series(ts_df.index)
            .between(
                pd.Timestamp(year=year, month=start_month, day=1),
                # get latest moment of the latest day of the month
                # ACHTUNG: this won't work if `end_month` is 12 (see commented code
                # below)
                pd.Timestamp(
                    dt.datetime.combine(
                        dt.date(year, end_month + 1, 1) - dt.timedelta(days=1),
                        dt.time.max,
                    ),
                ),
            )
            .values
        ]
        .resample("h")
        .mean()
        for year in range(start_year, end_year + 1)
    ]
)
# rename columns to remove "Log_" prefix
# ts_df = ts_df.rename(columns=lambda col: col.split("_")[-1])
# add columns axis title
ts_df = ts_df.rename_axis(columns="station_id")
# show the data frame
ts_df.head()

station_id,Log_1,Log_10,Log_101,Log_102,Log_111,Log_112,Log_113,Log_116,Log_117,Log_124,...,Log_8,Log_80,Log_82,Log_83,Log_86,Log_87,Log_89,Log_9,Log_98,Log_99
time,,,,,,,,,,,,,,,,,,,,,
2025-06-01 00:00:00,18.432771,17.569810,16.150085,16.733997,17.432288,17.386002,17.755843,17.137662,NaN,17.953002,...,17.955227,16.947178,16.886985,17.158579,17.394458,16.106470,15.643613,17.009486,15.816739,15.761997
2025-06-01 01:00:00,17.777206,17.289425,15.754877,16.280486,17.180832,16.786959,17.337936,16.371722,NaN,17.649475,...,17.750502,16.658783,16.511025,16.746904,17.130541,15.700802,16.089113,16.839030,15.749536,16.149640
2025-06-01 02:00:00,17.464777,17.393123,15.641388,16.381514,17.187508,16.990794,17.167926,16.314310,NaN,17.361524,...,17.957453,16.782063,16.303629,17.229343,17.476571,15.715489,16.029030,16.562206,16.507909,16.637198
2025-06-01 03:00:00,16.807876,17.359299,15.438443,16.270695,17.008151,16.369052,16.747794,15.977404,NaN,17.279634,...,17.847079,16.693497,16.013231,17.155019,17.184393,15.485618,15.585311,15.866585,16.346354,16.366382
2025-06-01 04:00:00,16.648992,16.980557,15.110437,15.977404,16.708184,16.292058,16.539953,15.549706,NaN,16.755805,...,17.493261,16.191030,15.875041,16.845038,16.863953,15.360113,15.554602,15.436217,16.044162,16.145635


In [ ]:
# region_gser = gpd.read_file(spatial_extent_filepath)["geometry"]
stations_gdf = pd.read_csv(stations_gdf_filepath, sep=";", encoding="latin1")
stations_gdf = gpd.GeoDataFrame(
    stations_gdf,
    geometry=gpd.points_from_xy(
        stations_gdf[stations_x_col], stations_gdf[stations_y_col]
    ),
    crs=stations_src_crs,
).to_crs(dst_crs)  # .to_crs(region_gser.crs)
# # filter by spatial extent
# stations_gdf = stations_gdf[stations_gdf.within(region_gser.iloc[0])]
# rename id column and set it as index
stations_gdf = stations_gdf.rename(columns={stations_id_col: "station_id"}).set_index(
    "station_id"
)
stations_gdf.index = stations_gdf.index.astype(str)
# show the geo-data frame
stations_gdf.head()

,Location,Longitude,Latitude,LV_95_E,LV_95_N,HuM,geometry
station_id,,,,,,,
Log_1,VonRoll,7.42252,46.95291,2598773.471,1200203.276,553.9,POINT (2598773.471 1200203.276)
Log_2,Europaplatz,7.40633,46.94333,2597540.582,1199138.657,555.5,POINT (2597540.582 1199138.657)
Log_7,Brunnhof,7.42713,46.94479,2599124.256,1199300.504,528.6,POINT (2599124.256 1199300.504)
Log_8,Wankdorf ESP,7.46247,46.96800,2601814.111,1201880.959,550.5,POINT (2601814.111 1201880.959)
Log_9,Bremgartenfriedhof,7.42015,46.95030,2598592.987,1199913.163,552.4,POINT (2598592.987 1199913.163)


In [ ]:
# save only stations for which we have both data and metadata
valid_stations = ts_df.columns.intersection(stations_gdf.index)
# save the time series of measurements to a file
ts_df[valid_stations].to_csv(dst_ts_df_filepath)
# save the stations' locations to a file
stations_gdf.loc[valid_stations].rename_axis(index="station_id").to_file(
    dst_stations_gdf_filepath
)